In [1]:
!pip install tardis-dev

  Attempting uninstall: aiofiles
    Found existing installation: aiofiles 25.1.0
    Uninstalling aiofiles-25.1.0:
      Successfully uninstalled aiofiles-25.1.0


In [2]:
!pip uninstall -y tardis-dev
!pip install tardis-dev==2.1.3

Found existing installation: tardis-dev 4.2.1
Uninstalling tardis-dev-4.2.1:
  Successfully uninstalled tardis-dev-4.2.1


In [3]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
import asyncio
import polars as pl
from tardis_dev import datasets
import os

def download_and_convert():
    # 1. Descargar datos
    # Quitamos el 'await' porque la librería ya gestiona la descarga
    datasets.download(
        exchange="binance-futures",
        data_types=["book_snapshot_25"],
        from_date="2026-04-01",
        to_date="2026-04-02",
        symbols=["BTCUSDT"]
    )

    # 2. Convertir a Parquet
    if not os.path.exists("datasets"):
        print("La carpeta 'datasets' no se creó. Revisa la descarga.")
        return

    for file in os.listdir("datasets"):
        if file.endswith(".csv.gz"):
            path_csv = os.path.join("datasets", file)
            path_parquet = path_csv.replace(".csv.gz", ".parquet")

            print(f"Transformando {file} a formato Parquet...")
            # Usamos scan_csv para eficiencia
            pl.scan_csv(path_csv).sink_parquet(path_parquet)
            print(f"✅ Creado: {path_parquet}")

# Ejecutamos la función normalmente
download_and_convert()

Transformando binance-futures_book_snapshot_25_2026-01-01_BTCUSDT.csv.gz a formato Parquet...
✅ Creado: datasets/binance-futures_book_snapshot_25_2026-01-01_BTCUSDT.parquet


In [5]:


# Buscamos el nombre del archivo parquet del libro
book_file = [f for f in os.listdir("datasets") if "book_snapshot_25" in f and f.endswith(".parquet")][0]
path_book = os.path.join("datasets", book_file)

# Cargamos una vista previa
df_book = pl.scan_parquet(path_book)



In [6]:
import os
import polars as pl

# 1. Buscamos el nombre del archivo parquet
book_file = [f for f in os.listdir("datasets") if "book_snapshot_25" in f and f.endswith(".parquet")][0]
path_book = os.path.join("datasets", book_file)

# 2. Cargamos la vista previa (LazyFrame)
df_book = pl.scan_parquet(path_book)

# 3. Columnas base que siempre queremos mantener (ajusta si faltase alguna)
columnas_base = ["exchange", "symbol", "timestamp", "local_timestamp"]

# 4. Generamos dinámicamente los nombres de los 10 primeros niveles (del 0 al 9)
columnas_niveles = []
for i in range(10):
    columnas_niveles.append(f"bids[{i}].price")
    columnas_niveles.append(f"bids[{i}].amount")
    columnas_niveles.append(f"asks[{i}].price")
    columnas_niveles.append(f"asks[{i}].amount")

# Juntamos solo las columnas base que realmente existan en el archivo con los 10 niveles
todas_las_columnas = [col for col in columnas_base if col in df_book.columns] + columnas_niveles

# 5. Aplicamos el filtro en Polars
df_book_10 = df_book.select(todas_las_columnas)
df_book_10= df_book_10.drop(["exchange", "symbol"])

/tmp/ipykernel_4213/4101919769.py:23: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  todas_las_columnas = [col for col in columnas_base if col in df_book.columns] + columnas_niveles


In [8]:
df_book_10.collect().write_parquet('df_book_10.parquet')